# Deconvolve Positive Control Genes and plot
February 8, 2024

This notebook is designed to simplify the deconvolution process for gene expression and chromatin in one pass. The functions save the output to disk as plots, and eventually as data and metadata for downstream analysis.


In [3]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
def deconvolve_gene(gene_name, replicate, save_dir):
    """
    Deconvolve a gene's gene expression and chromatin
    """
    from src.model import Model
    from cc_src.chromatin_model import ChromatinModel
    from src.config import load_yl_replicate1_rg1_alpha_vst_config, \
        load_yl_replicate2_rg1_alpha_vst_config
    
    if replicate == 1:
        config = load_yl_replicate1_rg1_alpha_vst_config()
    elif replicate == 2:
        config = load_yl_replicate2_rg1_alpha_vst_config()
    else:
        raise ValueError("Undefined replicate")
    
    print(f"Deconvolving {gene_name}, replicate={replicate}")
    print(f"Deconvolving gene expression...", end="")
    ge_model = Model(config, gene_name)
    ge_model.deconvolve_find_optimal_gamma()
    print("Done.")

    print(f"Deconvolving chromatin...", end="")
    chrom_model = ChromatinModel(config)
    chrom_model.load_mnase_gene(gene_name, replicate=config.replicate)
    chrom_model.deconvolve_find_optimal_gamma()
    print("Done.")
    
    save_name = f"{save_dir}/{gene_name}_rep{config.replicate}_chromatin.png"
    fig = chrom_model.create_deconvolution_plots_abbreviated_flipped(ge_model=ge_model)
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)

    save_name = f"{save_dir}/{gene_name}_{config.replicate}_predicted.png"
    fig = chrom_model.plot_prediction_comparison()
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)


In [1]:
from cc_src.geneset import positive_control_genes
from src.timer import Timer

def save_replicate_deconvolutions(genes, save_dir1, save_dir2):
    save_dirs = {
        1: save_dir1,
        2: save_dir2
    }

    # For each replicate, deconvolve the positive control genes
    # and save the resulting figures to disk
    # TODO: Save the deconvolved data and metadata to disk
    #       Can use existing code for this, but requires some cleanup
    #       and simplification
    timer = Timer()


    total = len(genes)*2
    count = 0
    for replicate in [1, 2]:
        save_dir = save_dirs[replicate]
        for gene_name in genes:
            deconvolve_gene(gene_name, replicate, save_dir)
            count += 1
            print(f"\n===== Progress {count}/{total} - {timer.get_time()}\n\n")


In [ ]:
save_dir1 = "output/positive_controls_chromatin/rep1"
save_dir2 = "output/positive_controls_chromatin/rep2"
genes = positive_control_genes()

In [6]:
from cc_src.geneset import cyclin_genes

save_dir1 = "output/positive_controls_chromatin/rep1"
save_dir2 = "output/positive_controls_chromatin/rep2"
genes = cyclin_genes()

save_replicate_deconvolutions(genes, save_dir1, save_dir2)

Deconvolving CLN1, replicate=1
Deconvolving gene expression...Done.
Deconvolving chromatin...Loading MNase reads for CLN1...Done.
The histogram shape around the TSS is: (3, 9)
The shape of the flattened grid to be deconvolved is: (16, 27)
Applying normalization using scaling matrix: output/mnase/rep1_len_scaling_3len_bins.csv
The shape of the flattened grid to be deconvolved is: (16, 27)
Running the find optimal gamma procedure...
  ... The base fitting norm (rn) with no smoothing (gamma=0) is: 0.7100
  ... Searching for an optimal gamma value in the boundaries: [0.0001, 0.0100]
  ...  search left, rn_goal = 0.7810, rate = 10.0
  ...   gm = 0.0050, rn = 0.8546, rate = 20.37, time = 00:00:40.38
  ...   gm = 0.0026, rn = 0.8037, rate = 13.20, time = 00:00:49.85
  ...   gm = 0.0013, rn = 0.7716, rate = 8.68, time = 00:01:00.81
  ...  search right, rn_goal = 1.0300, rate = 45.1
  ... rn range: [0.7716, 0.9386]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 0.7716

  ...  search left, rn_goal = 1.2763, rate = 6.7


/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 1.3838, rate = 15.67, time = 00:01:47.06
  ...   gm = 0.0026, rn = 1.2901, rate = 7.84, time = 00:02:33.17
  ...   gm = 0.0013, rn = 1.2509, rate = 4.56, time = 00:03:17.63
  ...  search right, rn_goal = 1.6749, rate = 40.0
  ... rn range: [1.2509, 1.4306]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 1.2509, rate = 4.56, time = 00:04:47.36
  ...   gm = 0.0022, rn = 1.2892, rate = 7.77, time = 00:05:33.37
  ...   gm = 0.0031, rn = 1.3215, rate = 10.46, time = 00:06:19.77
  ...   gm = 0.0039, rn = 1.3143, rate = 9.87, time = 00:07:06.44
  ...   gm = 0.0048, rn = 1.3441, rate = 12.35, time = 00:07:51.95
  ...   gm = 0.0057, rn = 1.3541, rate = 13.19, time = 00:08:38.09
  ...   gm = 0.0065, rn = 1.4236, rate = 19.00, time = 00:09:24.87
  ...   gm = 0.0074, rn = 1.4324, rate = 19.73, time = 00:10:12.67
  ...   gm = 0.0083, rn = 1.4236, rate = 19.00, time = 00:10:58.18
  ...   gm = 0.0091, rn = 1.4056, rate = 17.49, time = 00:11:44.97
  

  ...   gm = 0.0057, rn = 1.0485, rate = 26.04, time = 00:02:01.87
  ...   gm = 0.0065, rn = 1.0660, rate = 28.13, time = 00:02:11.26
  ...   gm = 0.0074, rn = 1.0862, rate = 30.56, time = 00:02:21.04
  ...   gm = 0.0083, rn = 1.1050, rate = 32.82, time = 00:02:30.86
  ...   gm = 0.0091, rn = 1.1214, rate = 34.79, time = 00:02:40.25
  ...   gm = 0.0100, rn = 1.1355, rate = 36.49, time = 00:02:50.07
The x_grad1 is: [0.03061914 0.02882562 0.02454746 0.02235206 0.02274363 0.02014025
 0.01881723 0.01949213 0.01760507 0.01527301 0.01412021]
The x_grad2 is: [-0.00179353 -0.00303584 -0.00323678 -0.00090191 -0.0011059  -0.0019632
 -0.00032406 -0.00060608 -0.00210956 -0.00174243 -0.00115281]
The curvature is: [7.54275077e-05 1.68473630e-04 5.95221322e-04 9.49856829e-04
 1.21275534e-03 1.60095498e-03 1.67953879e-03 1.65057229e-03
 3.66750522e-03 3.78179491e-03 1.77982854e-03]
YPR119W: ... final gamma = 0.00913
Time to find optimal gamma: 00:02:59.50
Deconvolved in : 00:03:00.14
The fitting norm 

/Users/trung/opt/anaconda3/envs/cell-cycle-deconvolution/lib/python3.8/site-packages/cvxpy/problems/problem.py:1403: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


  ...   gm = 0.0050, rn = 1.9945, rate = 4.54, time = 00:01:46.76
  ...   gm = 0.0026, rn = 1.9471, rate = 2.06, time = 00:02:33.22
  ...   gm = 0.0038, rn = 1.9644, rate = 2.96, time = 00:03:19.79
  ...  search right, rn_goal = 2.6711, rate = 40.0
  ... rn range: [1.9644, 2.0698]
  ... search gamma in [0.0038 0.0100] for elbow
  ...   gm = 0.0038, rn = 1.9644, rate = 2.96, time = 00:04:52.41
  ...   gm = 0.0044, rn = 1.9776, rate = 3.65, time = 00:05:38.18
  ...   gm = 0.0051, rn = 1.9930, rate = 4.46, time = 00:06:23.58
  ...   gm = 0.0057, rn = 1.9989, rate = 4.77, time = 00:07:08.71
  ...   gm = 0.0063, rn = 2.0164, rate = 5.69, time = 00:07:53.97
  ...   gm = 0.0069, rn = 2.0243, rate = 6.10, time = 00:08:39.44
  ...   gm = 0.0075, rn = 2.0408, rate = 6.97, time = 00:09:25.33
  ...   gm = 0.0081, rn = 2.0460, rate = 7.24, time = 00:10:10.39
  ...   gm = 0.0088, rn = 2.0584, rate = 7.89, time = 00:10:54.95
  ...   gm = 0.0094, rn = 2.0618, rate = 8.06, time = 00:11:40.16
  ...   gm

  ...   gm = 0.0057, rn = 0.4621, rate = 38.97, time = 00:02:04.20
  ...   gm = 0.0065, rn = 0.4755, rate = 42.99, time = 00:02:13.64
  ...   gm = 0.0074, rn = 0.4878, rate = 46.70, time = 00:02:23.37
  ...   gm = 0.0083, rn = 0.4992, rate = 50.13, time = 00:02:33.35
  ...   gm = 0.0091, rn = 0.5098, rate = 53.33, time = 00:02:43.19
  ...   gm = 0.0100, rn = 0.5209, rate = 56.66, time = 00:02:52.85
The x_grad1 is: [0.01537368 0.01459528 0.01276509 0.01161343 0.0116071  0.01253098
 0.01285614 0.01186512 0.01100855 0.01085344 0.01106906]
The x_grad2 is: [-7.78398184e-04 -1.30429523e-03 -1.49092726e-03 -5.78993633e-04
  4.58779027e-04  6.24519054e-04 -3.32930985e-04 -9.23794403e-04
 -5.05842509e-04  3.02566498e-05  2.15626512e-04]
The curvature is: [9.91164870e-05 2.10992153e-04 6.59559652e-04 8.77907912e-04
 8.19276802e-04 9.64900828e-04 9.17934533e-04 6.75755987e-04
 7.33011000e-04 1.00057391e-03 9.66307645e-04]
YAL040C: ... final gamma = 0.00913
Time to find optimal gamma: 00:03:02.63


The x_grad2 is: [-1.40052819e-03 -1.58699231e-03 -1.17227131e-03 -7.72876667e-04
 -6.33221542e-04  2.31087501e-05 -5.15221393e-05 -2.91933249e-04
 -1.06504936e-04 -1.45915638e-04 -2.21650189e-04]
The curvature is: [0.00011522 0.00026183 0.00074539 0.00095348 0.00160718 0.00210243
 0.00159347 0.00200173 0.00315591 0.00269942 0.00168516]
YGR108W: ... final gamma = 0.00827
Time to find optimal gamma: 00:27:05.59
Deconvolved in : 00:27:06.17
The fitting norm is 0.38, the smoothing norm is: 14.52
Done.
Saved figure to output/positive_controls_chromatin/rep2/CLB1_rep2_chromatin.png
Saved figure to output/positive_controls_chromatin/rep2/CLB1_2_predicted.png

===== Progress 15/18 - 02:12:52.13


Deconvolving CLB2, replicate=2
Deconvolving gene expression...Done.
Deconvolving chromatin...Loading MNase reads for CLB2...Done.
The histogram shape around the TSS is: (3, 9)
The shape of the flattened grid to be deconvolved is: (15, 27)
Applying normalization using scaling matrix: output/mnase/rep2_

Saved figure to output/positive_controls_chromatin/rep2/CLB4_2_predicted.png

===== Progress 18/18 - 02:42:06.28


